In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.10.0 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("silver_taxi")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "password123")
    .getOrCreate()
)

spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

DataFrame[]

In [3]:
bronze = spark.table("lakehouse.taxi.bronze_trips")

zones = (
    spark.read.parquet("/home/jovyan/project/data/taxi_zone_lookup.parquet")
    .select(
        F.col("LocationID").cast("int").alias("zone_location_id"),
        F.col("Borough").alias("pickup_borough"),
        F.col("Zone").alias("pickup_zone"),
        F.col("service_zone").alias("pickup_service_zone")
    )
)

silver = (
    bronze
    .withColumn("trip_id", F.col("trip_id").cast("long"))
    .withColumn("VendorID", F.col("VendorID").cast("int"))
    .withColumn("tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("passenger_count", F.col("passenger_count").cast("double"))
    .withColumn("trip_distance", F.col("trip_distance").cast("double"))
    .withColumn("PULocationID", F.col("PULocationID").cast("int"))
    .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge").cast("double"), F.lit(0.0)))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn(
        "duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .withColumn("duration_hours", F.col("duration_minutes") / 60.0)
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("tpep_pickup_datetime").isNotNull())
    .filter(F.col("tpep_dropoff_datetime").isNotNull())
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("duration_minutes") > 0)
    .filter(F.col("duration_minutes") <= 240)
    .filter(F.col("PULocationID").isNotNull())
    .filter(F.col("DOLocationID").isNotNull())
    .join(zones, F.col("PULocationID") == F.col("zone_location_id"), "left")
    .withColumn("speed_mph", F.col("trip_distance") / F.col("duration_hours"))
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .filter(F.col("speed_mph") > 0)
    .filter(F.col("speed_mph") <= 100)
)

silver.writeTo("lakehouse.taxi.silver_trips").createOrReplace()

print("Silver taxi table created.")
spark.sql("SELECT COUNT(*) AS silver_taxi_count FROM lakehouse.taxi.silver_trips").show()

spark.sql("""
SELECT
    trip_id,
    pickup_date,
    pickup_hour,
    pickup_zone,
    trip_distance,
    duration_minutes,
    speed_mph,
    fare_amount,
    congestion_surcharge,
    fare_per_mile
FROM lakehouse.taxi.silver_trips
LIMIT 10
""").show(truncate=False)

Silver taxi table created.
+-----------------+
|silver_taxi_count|
+-----------------+
|          6554853|
+-----------------+

+-------+-----------+-----------+-----------------------------+-------------+------------------+------------------+-----------+--------------------+------------------+
|trip_id|pickup_date|pickup_hour|pickup_zone                  |trip_distance|duration_minutes  |speed_mph         |fare_amount|congestion_surcharge|fare_per_mile     |
+-------+-----------+-----------+-----------------------------+-------------+------------------+------------------+-----------+--------------------+------------------+
|0      |2025-01-01 |0          |Sutton Place/Turtle Bay North|1.6          |8.35              |11.497005988023954|10.0       |2.5                 |6.25              |
|1      |2025-01-01 |0          |Upper East Side North        |0.5          |2.55              |11.764705882352942|5.1        |2.5                 |10.2              |
|2      |2025-01-01 |0          